## Módulo 1 - Sistema Baseado em Conhecimento para Gestão de Emergências

### 1. Introdução
**Contexto**

A Proteção Civil da cidade pretende um sistema de apoio à decisão para avaliar alertas ambientais em tempo real. Por exemplo, detetarpicos de calor, risco de incêndio, precipitação alta e vento forte.

**Objetivos de Aprendizagem**
1.	Reconhecer como a lógica de predicados e as regras “se-então” estruturam conhecimento;
2.	Aplicar raciocínio dedutivo e heurísticas para priorizar respostas;
3.	Compreender incerteza usando uma Rede Bayesiana simples.

**Tarefas**
1.	Definir pelo menos 10 regras de risco em lógica de predicados (“Se temperatura > 40 °C ∧ humidade < 20 % → risco_incêndio_alto”);
2.	Implementar um motor de inferência que, tendo por base um knowledge base, leia o dataset e indique ações recomendadas para cada caso presente no ficheiro.
3.	Criar uma Rede Bayesiana com 3-4 nós e usar inferência por enumeração para atualizar as probabilidades.



### 2. Configuração do Ambiente e Importação de Dados
    a. Importação de Bibliotecas

In [28]:
import sys
print(sys.executable) 
import pandas as pd
print(pd.__version__)
import json
from pathlib import Path

c:\Users\tomas\Documents\ISCTE\UCs\5.6 -  Intodução IA\IIA_Project\.venv\Scripts\python.exe
3.0.2


    b. Carregamento do Dataset

In [29]:
base_path = Path("outputs/preprocessed_for_rules.csv")
decisions_path = Path("outputs/preprocessing_decisions.csv")

if not base_path.exists():
    raise FileNotFoundError(
        "Falta outputs/preprocessed_for_rules.csv. Corre primeiro a célula final de export no eda1.ipynb."
    )

df = pd.read_csv(base_path, sep=';')
print(f"Dataset final carregado: {base_path}")
print(f"Shape: {df.shape[0]} linhas x {df.shape[1]} colunas")
display(df.head())

if decisions_path.exists():
    print("\nResumo de decisões de pré-processamento:")
    display(pd.read_csv(decisions_path).head(10))

Dataset final carregado: outputs\preprocessed_for_rules.csv
Shape: 10768 linhas x 19 colunas


,city,datetime,CO,NO2,O3,PM10,PM2.5,SO2,temperature_c,humidity_percent,wind_speed_kmh,precipitation_mm,C6H6,NOx,air_quality_good,year,month,datetime_parsed,CO_8h_avg
0,Lisboa,05/09/25 01:00,0.96,25.20,84.09,11.82,9.12,6.75,18.9,82.0,16.2,0.0,NaN,NaN,True,2025,9,2025-09-05 01:00:00,NaN
1,Lisboa,05/09/25 02:00,0.75,26.40,86.20,13.24,8.87,5.11,18.8,80.0,15.5,0.0,NaN,NaN,True,2025,9,2025-09-05 02:00:00,NaN
2,Lisboa,05/09/25 03:00,0.87,25.16,74.41,15.18,10.84,5.76,18.6,79.0,11.5,0.0,NaN,NaN,True,2025,9,2025-09-05 03:00:00,NaN
3,Lisboa,05/09/25 04:00,0.51,13.59,68.57,17.48,13.14,5.03,18.3,77.0,11.3,0.0,NaN,NaN,True,2025,9,2025-09-05 04:00:00,NaN
4,Lisboa,05/09/25 05:00,0.61,15.89,78.79,14.70,13.68,6.20,18.6,72.0,9.4,0.0,NaN,NaN,True,2025,9,2025-09-05 05:00:00,NaN



Resumo de decisões de pré-processamento:


,step,detail
0,drop_duplicates,removed=0
1,missing_threshold,threshold=75.0%
2,columns_over_threshold,"O3, PM10, PM2.5, SO2, pressure_hpa, wind_speed..."
3,columns_dropped,"pressure_hpa, wind_direction_deg, NMHC"
4,protected_features,"CO, CO_8h_avg, NO2, O3, PM10, PM2.5, SO2, air_..."
5,rule_features_detected,"CO_8h_avg, NO2, O3, PM10, PM2.5, SO2, humidity..."
6,row_policy_for_rules,strict_complete_cases_removed=9336
7,co_8h_avg,"rolling=8, min_periods=6, by=city"
8,outlier_method,"IQR (1.5*IQR), detection only"
9,normalization,z-score columns with std>0 (analysis copy only)


### 3. Definição da Base de Conhecimento (regras.json)

In [30]:
rules_path = Path("regras.json")
if not rules_path.exists():
    raise FileNotFoundError("Falta regras.json em Module_1")

with rules_path.open("r", encoding="utf-8") as f:
    rules_payload = json.load(f)

rules = rules_payload.get("rules", []) if isinstance(rules_payload, dict) else rules_payload
print(f"Total de regras: {len(rules)}")
display(pd.DataFrame([{
    "id": r.get("id"),
    "description": r.get("description"),
    "priority": r.get("priority"),
    "risk_level": (r.get("consequence") or {}).get("risk_level", r.get("risk_level"))
} for r in rules]))

Total de regras: 12


,id,description,priority,risk_level
0,R01_NO2_ALTO,Alerta NO2 critico - limite horario UE,10,ALTO
1,R02_NO2_MODERADO,Alerta NO2 preventivo,5,MODERADO
2,R03_PM10_ALTO,Particulas inalaveis PM10 excedem limite 24h,9,ALTO
3,R04_PM25_ALTO,Particulas finas PM2.5 excedem limite anual UE,10,ALTO
4,R05_O3_ALTO,Ozono troposferico - limiar de informacao,8,ALTO
5,R06_CO_ALTO,Monoxido de carbono - media 8h critica,9,ALTO
6,R07_SO2_ALTO,Dioxido de enxofre - limite horario,7,ALTO
7,R08_CALOR_EXTREMO,Onda de calor extremo,10,ALTO
8,R09_RISCO_INCENDIO,Risco maximo de incendio florestal,10,ALTO
9,R10_VENTO_FORTE,Vento forte - alerta laranja,7,ALTO


### 4. Implementação do Motor de Inferência (rules_engine.py)

In [31]:
import subprocess

rules_output = Path("outputs/rules_inference_output.csv")
cmd = [
    sys.executable,
    "rules_engine.py",
    "--input", str(base_path),
    "--rules", "regras.json",
    "--output", str(rules_output),
]

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Falha na execução de rules_engine.py")

print(f"Output gerado: {rules_output}")

Processed rows: 10768
Output written to: outputs\rules_inference_output.csv
Risk distribution:
  high: 1009
  moderate: 3828
  none: 5931
Most triggered rules:
  R02_NO2_MODERADO: 4059
  R01_NO2_ALTO: 398
  R09_RISCO_INCENDIO: 388
  R04_PM25_ALTO: 140
  R12_QUALIDADE_AR_PESSIMA: 131
  R08_CALOR_EXTREMO: 61
  R03_PM10_ALTO: 7
  R11_PRECIPITACAO_INTENSA: 2

Output gerado: outputs\rules_inference_output.csv


### 5. Execução e Análise de Resultados

In [32]:
rules_df = pd.read_csv(rules_output, sep=';')
print(f"Resultados do motor de regras: {rules_df.shape[0]} linhas")

risk_dist = rules_df["overall_risk"].value_counts(dropna=False).rename_axis("overall_risk").to_frame("n")
risk_dist["pct"] = (risk_dist["n"] / len(rules_df) * 100).round(2)
display(risk_dist)

cols_to_show = [
    c for c in ["city", "datetime", "overall_risk", "matched_rule_ids", "recommended_actions"]
    if c in rules_df.columns
]
display(rules_df[cols_to_show].head(10))

Resultados do motor de regras: 10768 linhas


,n,pct
overall_risk,,
none,5931,55.08
moderate,3828,35.55
high,1009,9.37


,city,datetime,overall_risk,matched_rule_ids,recommended_actions
0,Lisboa,05/09/25 01:00,none,NaN,NaN
1,Lisboa,05/09/25 02:00,none,NaN,NaN
2,Lisboa,05/09/25 03:00,none,NaN,NaN
3,Lisboa,05/09/25 04:00,none,NaN,NaN
4,Lisboa,05/09/25 05:00,none,NaN,NaN
5,Lisboa,05/09/25 06:00,none,NaN,NaN
6,Lisboa,05/09/25 07:00,high,R04_PM25_ALTO,Grupos sensiveis permanecam em ambientes inter...
7,Lisboa,05/09/25 08:00,none,NaN,NaN
8,Lisboa,05/09/25 09:00,none,NaN,NaN
9,Lisboa,05/09/25 10:00,high,R04_PM25_ALTO,Grupos sensiveis permanecam em ambientes inter...


### 6. Modelagem de Incerteza: Rede Bayesiana (bayes_alerts.py)

In [33]:
bayes_output = Path("outputs/bayes_inference_output.csv")
cmd = [
    sys.executable,
    "bayes_alerts.py",
    "--input", str(base_path),
    "--output", str(bayes_output),
]

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Falha na execução de bayes_alerts.py")

print(f"Output gerado: {bayes_output}")

Processed rows: 10768
Output written to: outputs\bayes_inference_output.csv

Output gerado: outputs\bayes_inference_output.csv


### 7. Inferência por Enumeração

In [34]:
bayes_df = pd.read_csv(bayes_output, sep=';')
print(f"Resultados Bayes: {bayes_df.shape[0]} linhas")

for col in ["p_fire_risk_true", "p_fire_risk_false"]:
    if col in bayes_df.columns:
        bayes_df[col] = pd.to_numeric(bayes_df[col], errors="coerce")

summary_bayes = bayes_df[["p_fire_risk_true", "p_fire_risk_false"]].describe().T
display(summary_bayes)

if "p_fire_risk_true" in bayes_df.columns:
    top_cols = [c for c in ["city", "datetime", "p_fire_risk_true"] if c in bayes_df.columns]
    display(bayes_df[top_cols].sort_values("p_fire_risk_true", ascending=False).head(10))

Resultados Bayes: 10768 linhas


,count,mean,std,min,25%,50%,75%,max
p_fire_risk_true,10768.0,0.230479,0.232923,0.050,0.125,0.125,0.125,0.875
p_fire_risk_false,10768.0,0.769521,0.232923,0.125,0.875,0.875,0.875,0.950


,city,datetime,p_fire_risk_true
2736,UCI_Dataset,03/05/04 16:00,0.875
2735,UCI_Dataset,03/05/04 15:00,0.875
2734,UCI_Dataset,03/05/04 14:00,0.875
3049,UCI_Dataset,16/05/04 17:00,0.875
3048,UCI_Dataset,16/05/04 16:00,0.875
3047,UCI_Dataset,16/05/04 15:00,0.875
3045,UCI_Dataset,16/05/04 13:00,0.875
3095,UCI_Dataset,18/05/04 15:00,0.875
3094,UCI_Dataset,18/05/04 14:00,0.875
3093,UCI_Dataset,18/05/04 13:00,0.875


### 8. Conclusão Crítica e Ética

### Notas Finais

- Existe uma única base oficial para a pipeline: `outputs/preprocessed_for_rules.csv`.
- Motor de regras e rede Bayesiana correm sobre a mesma base para consistência.
- Outliers não são removidos automaticamente (podem representar eventos críticos).
- O critério de limpeza fica rastreado em `outputs/preprocessing_decisions.csv`.